## Lab 7.2: Stereo

In this lab you will implement a simple stereo block-matching algorithm to estimate a depth map from a stereo pair.

In [ ]:
!wget -nc -q -O bennu_left.png "https://www.dropbox.com/scl/fi/8uwlu1schqh4uc6zrnshm/bennu_left.png?rlkey=eq3v9ynr2loeduti9evzjp3xe&dl=1"
!wget -nc -q -O bennu_right.png "https://www.dropbox.com/scl/fi/om4lra2cj6p31dpzhgu5y/bennu_right.png?rlkey=7z0aapzc7vk3gu87n272uxz6r&dl=1"
!wget -nc -q -O plushies_left.png "https://www.dropbox.com/scl/fi/k41xm8f6uitbrarjevcl2/plushies_left.png?rlkey=r13ie5trfpkvbvgc08exqiidk&dl=1"
!wget -nc -q -O plushies_right.png "https://www.dropbox.com/scl/fi/k7uaeftf4izh2csr3bu2v/plushies_right.png?rlkey=bxq17zvqhp9li0crw1rjg3li5&dl=1"
!wget -nc -q -O toys_left.png "https://www.dropbox.com/scl/fi/t501h805wsfaiit9wt0rf/toys_left.png?rlkey=nr5ex7dbhlp4t9njofjj1kt5x&dl=1"
!wget -nc -q -O toys_right.png "https://www.dropbox.com/scl/fi/4fzptyxva4bp16552ly81/toys_right.png?rlkey=wltf8qe8hgoft9tixuy15d6ui&dl=1"

In [ ]:
import matplotlib.pyplot as plt

import numpy as np
import imageio
import skimage
from skimage.color import rgb2gray

from scipy.ndimage import convolve

Let's load our Bennu stereo pair.  By inspecting the images you can see that as you go from the left image to the right image, the points in the scene move to the left, as you would expect.

In [ ]:
left = rgb2gray(imageio.imread('bennu_left.png'))
right = rgb2gray(imageio.imread('bennu_right.png'))

fig,axes = plt.subplots(1,2)
axes[0].imshow(left,cmap='gray')
axes[1].imshow(right,cmap='gray')

Now you will write a simple stereo block matching algorithm to estimate the disparity of each pixel in the left image (how far the pixel moved to the left in the right image).

First choose a maximum disparity -- I chose a maximum of 35 pixels.  

Make a cost volume to store the results.  The cost volume will have size $H \times W \times D$ where $H \times W$ is the size of the images and and $D$ is the number of disparity settings.  Initialize the cost volume to have a value of infinity at all locations.

What we will do is shift the left image by each disparity setting and compare it to the right image.

For each disparity setting $d$ from 0 to the maximum, do the following:
- Take columns $d$ to $W-1$ in the left image and columns $0$ to $W-d-1$ in the right image.
- Compute the squared difference of the two cropped images.
- Convolve the squared difference with a box filter (a square filter of all ones).  (I used a box filter size of $5\times5$.)
- Store the result in columns $d$ to $W-1$ in layer $d$ of the cost volume.

Finally, compute the best disparity at each pixel by finding the disparity setting with lowest cost at each pixel.  (See `np.argmin`.)  The result is your disparity map.  `imshow` the disparity and verify that it matches the depth of the scene.

Try your algorithm on the Bennu stereo pair as well as "plushies" and "toys."  (Note that the Bennu stereo pair seems to have been shifted so that the closer points actually have lower disparity.)